# Spark Session Initialization

Initialize the Spark Session used for all DataFrame operations in this notebook.

In [2]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
import os
import sys

builder = ( SparkSession.builder \
    .appName("BGG Data Validation") \
    .master("local[*]") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.delta.logStore.class", "org.apache.spark.sql.delta.storage.LocalLogStore")
            
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

# Paths Configuration

Define all input and output paths used in this notebook.

In [4]:
from pathlib import Path
from pyspark.sql.functions import (
    col, year, month, dayofmonth,
    round as spark_round, to_date,
    current_timestamp, when, round,
    months_between
)

PROJECT_ROOT = Path.cwd().parents[0]
DATA_PATH = PROJECT_ROOT / "data"

SILVER_PATH = DATA_PATH / "silver"
GOLD_PATH = DATA_PATH / "gold"

## Add project root to sys.path to enable importing functions from utils
sys.path.append(str(Path().resolve().parent))

# Read Silver Data

Load enriched silver-level Delta tables into Spark DataFrames for aggregation.

In [5]:
sales_enriched_df = spark.read.format("delta").load(str(SILVER_PATH / "sales_enriched"))

# Aggregate and Create Gold Data

Perform aggregations and transformations to generate gold-level analytical datasets.

In [6]:
from pyspark.sql.functions import (
     sum, countDistinct, round, avg, 
    row_number, dense_rank, concat_ws
    )

monthly_sales_summary_df = (
    sales_enriched_df
    .groupBy("year", "month")
    .agg(
        round(sum("total_revenue"),2).alias("total_revenue"),
        round(sum("total_profit"),2).alias("total_profit"),
        countDistinct("sale_id").alias("total_orders")
    )
    .withColumn("gold_generated_timestamp", current_timestamp())
)

In [7]:
country_sales_summary_df = (
    sales_enriched_df
    .groupBy("customer_country_name")
    .agg(
        round(sum("total_revenue"),2).alias("total_revenue"),
        round(avg("profit_margin_pct"),2).alias("avg_profit_margin_pct")
    )
    .withColumn("gold_generated_timestamp", current_timestamp())
)

In [8]:
from pyspark.sql.window import Window

window_spec = Window.partitionBy("year", "month") \
                    .orderBy(col("montly_total_revenue").desc())

top_10_montly_game_sales_df = (
    sales_enriched_df
    .groupBy("year", "month","game_name")
    .agg(round(sum("total_revenue"),2).alias("montly_total_revenue"))
    .withColumn("rank",dense_rank().over(window_spec))
    .filter(col("rank") <= 10)
    .withColumn("gold_generated_timestamp", current_timestamp())
)

In [10]:
employee_sales_df = (
    sales_enriched_df
    .withColumn("employee_name",concat_ws(" ", col("employee_first_name"), col("employee_last_name")))
    .groupBy("year", "month", "employee_name")
    .agg(
        round(sum("total_revenue"),2).alias("total_sales"),
        round(sum("total_profit"),2).alias("total_profit")
    )
    .withColumn("gold_generated_timestamp", current_timestamp())
)

# Save to Gold

Persist the gold-level DataFrames to Delta format for reporting and BI consumption.

In [11]:
# from pyspark.sql import DataFrame

# def save_to_gold(
#     df: DataFrame,
#     table_name: str,
#     path: Path = GOLD_PATH,
#     mode: str = "overwrite") -> None:

#     output_path = path / table_name

#     (
#         df.write
#         .format("delta")
#         .mode(mode)
#         .save(str(output_path))
#     )

#     print(f"Saved to gold layer delta table: {output_path}")

In [12]:
from utils.data_io import save_to_gold

save_to_gold(monthly_sales_summary_df, "monthly_sales_summary")
save_to_gold(country_sales_summary_df, "country_sales")
save_to_gold(top_10_montly_game_sales_df, "top_10_games")
save_to_gold(employee_sales_df, "employee_performance") 

Saved to gold layer delta table: D:\Python\Project_Boardgames_Data_Validation\data\gold\monthly_sales_summary
Saved to gold layer delta table: D:\Python\Project_Boardgames_Data_Validation\data\gold\country_sales
Saved to gold layer delta table: D:\Python\Project_Boardgames_Data_Validation\data\gold\top_10_games
Saved to gold layer delta table: D:\Python\Project_Boardgames_Data_Validation\data\gold\employee_performance
